In [1]:
import networkx as nx
import pandas as pd
import numpy as np
from rdflib import URIRef
import pickle

## Process NP-KG and reference data for embeddings pipeline

In [2]:
FILE_PATH = '/home/sanya/npkg-workspace/reference-data/db_export/combined_interactions_20240904.tsv'
DIR_OUT = 'data/reference-data/'

In [3]:
refdf = pd.read_csv(FILE_PATH, sep='\t')
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1431 entries, 0 to 1430
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1431 non-null   object 
 1   np_id              1431 non-null   object 
 2   np                 1431 non-null   object 
 3   drug_id            1431 non-null   object 
 4   drug               1431 non-null   object 
 5   interaction_type   1431 non-null   object 
 6   source_data        1431 non-null   object 
 7   protein_target     654 non-null    object 
 8   adr_label          216 non-null    object 
 9   adr_uri            216 non-null    object 
 10  llm-interact-type  1431 non-null   object 
 11  mech-type          1431 non-null   object 
 12  adr                39 non-null     object 
 13  related_np         1431 non-null   object 
 14  related_np_id      1431 non-null   object 
 15  napdi_id           170 non-null    float64
 16  fideo_id           520 n

In [4]:
refdf.loc[refdf['interaction_type'].isna()]

,id,np_id,np,drug_id,drug,interaction_type,source_data,protein_target,adr_label,adr_uri,...,related_np,related_np_id,napdi_id,fideo_id,natmed_id,ddid_id,stockley_id,overall_effect,reference,evidence_type


In [5]:
refdf['interaction_type'] = refdf['interaction_type'].str.lower()
refdf['interaction_type'].value_counts()

positive    1190
negative     155
both          86
Name: interaction_type, dtype: int64

In [6]:
##import graph
KG_PATH = '/home/sanya/np-kg/resources/knowledge_graphs/'
NodeLabelsFile = KG_PATH + 'nodeLabels_v3.0.0.pickle'
#KG_NAME_MERGED = 'NP-KG_v2.0.0.gpickle'
KG_NAME_MERGED_WEIGHTED = 'NP-KG_BETWEENNESS-WEIGHTED-PK-HASHED_v3.0.0.gpickle'

In [7]:
with open(KG_PATH+KG_NAME_MERGED_WEIGHTED, 'rb') as f:
    npkg = pickle.load(f)

In [8]:
with open(NodeLabelsFile, 'rb') as filep:
    nodeLabels = pickle.load(filep)

In [9]:
def get_graph_stats(kg):
    nodes = nx.number_of_nodes(kg)
    edges = nx.number_of_edges(kg)
    self_loops = nx.number_of_selfloops(kg)

    print('There are {} nodes, {} edges, and {} self-loop(s)'.format(nodes, edges, self_loops))
    # get degree information
    avg_degree = float(edges) / nodes
    print('The Average Degree is {}'.format(avg_degree))
    
    print('Nodes with highest degree:')
    n_deg = sorted([(str(x[0]), x[1]) for x in  kg.degree], key=lambda x: x[1], reverse=1)[:6]

    for x in n_deg:
        print('Label: {}'.format(nodeLabels[x[0]]))
        print('{} (degree={})'.format(x[0], x[1]))
    # get network density
    density = nx.density(kg)

    print('The density of the graph is: {}'.format(density))

In [10]:
get_graph_stats(npkg)

There are 1089139 nodes, 7786937 edges, and 453 self-loop(s)
The Average Degree is 7.149626448047495
Nodes with highest degree:
Label: SNV
http://purl.obolibrary.org/obo/SO_0001483 (degree=214261)
Label: protein_coding_gene
http://purl.obolibrary.org/obo/SO_0001217 (degree=189350)
Label: transcript
http://purl.obolibrary.org/obo/SO_0000673 (degree=178309)
Label: Homo sapiens
http://purl.obolibrary.org/obo/NCBITaxon_9606 (degree=91837)
Label: protein
http://purl.obolibrary.org/obo/PR_000000001 (degree=56320)
Label: Mus musculus
http://purl.obolibrary.org/obo/NCBITaxon_10090 (degree=53597)
The density of the graph is: 6.5644816800511e-06


In [11]:
refdf.head()

,id,np_id,np,drug_id,drug,interaction_type,source_data,protein_target,adr_label,adr_uri,...,related_np,related_np_id,napdi_id,fideo_id,natmed_id,ddid_id,stockley_id,overall_effect,reference,evidence_type
0,napdi_88|napdi_89,http://purl.obolibrary.org/obo/CHEBI_16118,berberine|berberine,http://purl.obolibrary.org/obo/CHEBI_16469,17-beta-estradiol|17-beta-estradiol,positive,NaPDI|NaPDI,CYP1A1|CYP1A2,NaN,NaN,...,Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...,89.0,NaN,NaN,NaN,NaN,In Vitro Enzyme Inhibition,PMID:26403084,In Vitro Enzyme Inhibition
1,napdi_138,http://purl.obolibrary.org/obo/CHEBI_4806,epigallocatechin gallate,http://purl.obolibrary.org/obo/CHEBI_16469,17-beta-estradiol,positive,NaPDI,UGT1A1,NaN,NaN,...,Camellia sinensis,http://napdi.org/napdi_srs_imports:camellia_si...,138.0,NaN,NaN,NaN,NaN,In Vitro Enzyme Inhibition,PMID:20666626,In Vitro Enzyme Inhibition
2,napdi_9,http://purl.obolibrary.org/obo/CHEBI_69919,(-)-beta-hydrastine,http://purl.obolibrary.org/obo/CHEBI_791,17-beta-estradiol glucuronide,positive,NaPDI,OATP1B1 (SLCO1B1),NaN,NaN,...,Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...,9.0,NaN,NaN,NaN,NaN,In Vitro Transporter Inhibition,PMID:nan,In Vitro Transporter Inhibition
3,napdi_219,http://napdi.org/napdi_srs_imports:hydrastis_c...,goldenseal root extract,http://purl.obolibrary.org/obo/CHEBI_791,17-beta-estradiol glucuronide,positive,NaPDI,OATP1B1 (SLCO1B1),NaN,NaN,...,Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...,219.0,NaN,NaN,NaN,NaN,In Vitro Transporter Inhibition,PMID:nan,In Vitro Transporter Inhibition
4,napdi_15,http://purl.obolibrary.org/obo/CHEBI_90,(-)-epicatechin,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone,negative,NaPDI,UGT,NaN,NaN,...,Camellia sinensis,http://napdi.org/napdi_srs_imports:camellia_si...,15.0,NaN,NaN,NaN,NaN,In Vitro Enzyme Negligible Inhibition,PMID:29467215,In Vitro Enzyme Inhibition


In [12]:
refdf['npkg_edge'] = pd.NA
refdf['npkg_edge_label'] = pd.NA

In [13]:
for i in range(len(refdf.index)):
    np_id = refdf.loc[i, 'np_id']
    drug_id = refdf.loc[i, 'drug_id']
    source = URIRef(np_id)
    target = URIRef(drug_id)
    if source in npkg and target in npkg:
        if npkg.has_edge(source, target):
            edge_data = npkg.get_edge_data(source, target)
            refdf.at[i, 'npkg_edge'] = edge_data['key']
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1431 entries, 0 to 1430
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1431 non-null   object 
 1   np_id              1431 non-null   object 
 2   np                 1431 non-null   object 
 3   drug_id            1431 non-null   object 
 4   drug               1431 non-null   object 
 5   interaction_type   1431 non-null   object 
 6   source_data        1431 non-null   object 
 7   protein_target     654 non-null    object 
 8   adr_label          216 non-null    object 
 9   adr_uri            216 non-null    object 
 10  llm-interact-type  1431 non-null   object 
 11  mech-type          1431 non-null   object 
 12  adr                39 non-null     object 
 13  related_np         1431 non-null   object 
 14  related_np_id      1431 non-null   object 
 15  napdi_id           170 non-null    float64
 16  fideo_id           520 n

In [14]:
refdf.loc[refdf['npkg_edge'].notnull()]

,id,np_id,np,drug_id,drug,interaction_type,source_data,protein_target,adr_label,adr_uri,...,napdi_id,fideo_id,natmed_id,ddid_id,stockley_id,overall_effect,reference,evidence_type,npkg_edge,npkg_edge_label
4,napdi_15,http://purl.obolibrary.org/obo/CHEBI_90,(-)-epicatechin,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone,negative,NaPDI,UGT,NaN,NaN,...,15.0,NaN,NaN,NaN,NaN,In Vitro Enzyme Negligible Inhibition,PMID:29467215,In Vitro Enzyme Inhibition,http://purl.obolibrary.org/obo/RO_0002213,<NA>
5,napdi_37,http://purl.obolibrary.org/obo/CHEBI_4806,(-)-epigallocatechin gallate,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone,positive,NaPDI,UGT,NaN,NaN,...,37.0,NaN,NaN,NaN,NaN,In Vitro Enzyme Inhibition,PMID:29467215,In Vitro Enzyme Inhibition,http://purl.obolibrary.org/obo/RO_0002449,<NA>
8,napdi_127,http://purl.obolibrary.org/obo/CHEBI_70255,epicatechin gallate,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone,positive,NaPDI,UGT,NaN,NaN,...,127.0,NaN,NaN,NaN,NaN,In Vitro Enzyme Inhibition,PMID:29467215,In Vitro Enzyme Inhibition,http://purl.obolibrary.org/obo/RO_0002449,<NA>
11,napdi_307|napdi_308|napdi_309,http://purl.obolibrary.org/obo/CHEBI_6956,mitragynine|mitragynine|mitragynine,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone|4-methylumbelliferone|4-...,negative,NaPDI|NaPDI|NaPDI,UGT|UGT1A1|UGT2B7,NaN,NaN,...,309.0,NaN,NaN,NaN,NaN,In Vitro Enzyme Negligible Inhibition,PMID:26692748,In Vitro Enzyme Inhibition,http://purl.obolibrary.org/obo/RO_0002449,<NA>
34,ddid_10|ddid_11|ddid_12|ddid_13,http://purl.obolibrary.org/obo/CHEBI_27732,Caffeine |Caffeine |Caffeine |Caffeine,http://purl.obolibrary.org/obo/CHEBI_16335,Adenosine|Adenosine|Adenosine|Adenosine,positive,FDI-Package Insert|FDI-Package Insert|FDI-Pack...,NaN,Seizures,http://purl.obolibrary.org/obo/HP_0001250,...,NaN,NaN,NaN,13.0,NaN,positive,PMID:nan,NaN,http://purl.obolibrary.org/obo/RO_0002434|http...,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1297,napdi_318|napdi_320,http://purl.obolibrary.org/obo/CHEBI_6956,mitragynine|mitragynine,http://purl.obolibrary.org/obo/CHEBI_17347,testosterone|testosterone,negative,NaPDI|NaPDI,CYP3A4|CYP3A5,NaN,NaN,...,320.0,NaN,NaN,NaN,NaN,In Vitro Enzyme Negligible Inhibition,PMID:31707106,In Vitro Enzyme Inhibition,http://purl.obolibrary.org/obo/RO_0002449,<NA>
1303,natmed_241,http://napdi.org/napdi_srs_imports:camellia_si...,Green tea,http://purl.obolibrary.org/obo/CHEBI_28177,THEOPHYLLINE,positive,NatMedPro,NaN,NaN,NaN,...,NaN,NaN,241.0,NaN,NaN,NaN,NaN,Non-randomized controlled trial or case contro...,http://purl.obolibrary.org/obo/RO_0002180,<NA>
1304,natmed_270,http://napdi.org/napdi_srs_imports:paullinia_c...,Guarana,http://purl.obolibrary.org/obo/CHEBI_28177,THEOPHYLLINE,positive,NatMedPro,NaN,NaN,NaN,...,NaN,NaN,270.0,NaN,NaN,NaN,NaN,In vitro or animal study or theoretical study ...,http://purl.obolibrary.org/obo/RO_0002180,<NA>
1402,fideo_310|natmed_6|stockley_99,http://napdi.org/napdi_srs_imports:camellia_si...,tea food product|Green tea|Green Tea,http://purl.obolibrary.org/obo/CHEBI_10033,warfarin|WARFARIN (Coumadin)|Warfarin and rela...,both,FIDEO|NatMedPro|Stockleys,NaN,Abnormal bleeding|Decreased prothrombin time,http://purl.obolibrary.org/obo/HP_0001892|http...,...,NaN,310.0,6.0,NaN,99.0,NaN,"Taylor JR, Wilt VM. Probable antagonism of war...",Clinical Study|In vitro or animal study or the...,http://purl.obolibrary.org/obo/RO_0002449,<NA>


In [15]:
for i in range(len(refdf.index)):
    npkg_edge = refdf.loc[i, 'npkg_edge']
    if not pd.isna(npkg_edge):
        edge_list = npkg_edge.split('|')
        if len(edge_list) == 1:
            refdf.at[i, 'npkg_edge_label'] = nodeLabels.get(edge_list[0], '')
        else:
            edge_label_list = [nodeLabels.get(x, '') for x in edge_list]
            refdf.at[i, 'npkg_edge_label'] = '|'.join(edge_label_list)
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1431 entries, 0 to 1430
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1431 non-null   object 
 1   np_id              1431 non-null   object 
 2   np                 1431 non-null   object 
 3   drug_id            1431 non-null   object 
 4   drug               1431 non-null   object 
 5   interaction_type   1431 non-null   object 
 6   source_data        1431 non-null   object 
 7   protein_target     654 non-null    object 
 8   adr_label          216 non-null    object 
 9   adr_uri            216 non-null    object 
 10  llm-interact-type  1431 non-null   object 
 11  mech-type          1431 non-null   object 
 12  adr                39 non-null     object 
 13  related_np         1431 non-null   object 
 14  related_np_id      1431 non-null   object 
 15  napdi_id           170 non-null    float64
 16  fideo_id           520 n

In [16]:
refdf.to_csv(DIR_OUT+'reference_data_combined_interactions_nodrugclass_dedup_20240905_with_npkg_edges.tsv', sep='\t', index=False)

In [17]:
refdf_nokg = refdf.loc[refdf['npkg_edge'].isna()]
refdf_nokg = refdf_nokg.drop(columns=['npkg_edge', 'npkg_edge_label'])
refdf_nokg = refdf_nokg.reset_index(drop=True)
refdf_nokg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1347 entries, 0 to 1346
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1347 non-null   object 
 1   np_id              1347 non-null   object 
 2   np                 1347 non-null   object 
 3   drug_id            1347 non-null   object 
 4   drug               1347 non-null   object 
 5   interaction_type   1347 non-null   object 
 6   source_data        1347 non-null   object 
 7   protein_target     597 non-null    object 
 8   adr_label          207 non-null    object 
 9   adr_uri            207 non-null    object 
 10  llm-interact-type  1347 non-null   object 
 11  mech-type          1347 non-null   object 
 12  adr                39 non-null     object 
 13  related_np         1347 non-null   object 
 14  related_np_id      1347 non-null   object 
 15  napdi_id           138 non-null    float64
 16  fideo_id           513 n

In [18]:
refdf_nokg.to_csv(DIR_OUT+'reference_data_combined_interactions_nodrugclass_dedup_20240905_without_npkg_edges.tsv', sep='\t', index=False)

### Load data and convert to CURIEs -- save only edges for prediction

In [19]:
refdf = pd.read_csv(DIR_OUT+'reference_data_combined_interactions_nodrugclass_dedup_20240905_without_npkg_edges.tsv', sep='\t')

In [20]:
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1347 entries, 0 to 1346
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1347 non-null   object 
 1   np_id              1347 non-null   object 
 2   np                 1347 non-null   object 
 3   drug_id            1347 non-null   object 
 4   drug               1347 non-null   object 
 5   interaction_type   1347 non-null   object 
 6   source_data        1347 non-null   object 
 7   protein_target     597 non-null    object 
 8   adr_label          207 non-null    object 
 9   adr_uri            207 non-null    object 
 10  llm-interact-type  1347 non-null   object 
 11  mech-type          1347 non-null   object 
 12  adr                39 non-null     object 
 13  related_np         1347 non-null   object 
 14  related_np_id      1347 non-null   object 
 15  napdi_id           138 non-null    float64
 16  fideo_id           513 n

In [21]:
refdf.head()

,id,np_id,np,drug_id,drug,interaction_type,source_data,protein_target,adr_label,adr_uri,...,related_np,related_np_id,napdi_id,fideo_id,natmed_id,ddid_id,stockley_id,overall_effect,reference,evidence_type
0,napdi_88|napdi_89,http://purl.obolibrary.org/obo/CHEBI_16118,berberine|berberine,http://purl.obolibrary.org/obo/CHEBI_16469,17-beta-estradiol|17-beta-estradiol,positive,NaPDI|NaPDI,CYP1A1|CYP1A2,NaN,NaN,...,Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...,89.0,NaN,NaN,NaN,NaN,In Vitro Enzyme Inhibition,PMID:26403084,In Vitro Enzyme Inhibition
1,napdi_138,http://purl.obolibrary.org/obo/CHEBI_4806,epigallocatechin gallate,http://purl.obolibrary.org/obo/CHEBI_16469,17-beta-estradiol,positive,NaPDI,UGT1A1,NaN,NaN,...,Camellia sinensis,http://napdi.org/napdi_srs_imports:camellia_si...,138.0,NaN,NaN,NaN,NaN,In Vitro Enzyme Inhibition,PMID:20666626,In Vitro Enzyme Inhibition
2,napdi_9,http://purl.obolibrary.org/obo/CHEBI_69919,(-)-beta-hydrastine,http://purl.obolibrary.org/obo/CHEBI_791,17-beta-estradiol glucuronide,positive,NaPDI,OATP1B1 (SLCO1B1),NaN,NaN,...,Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...,9.0,NaN,NaN,NaN,NaN,In Vitro Transporter Inhibition,PMID:nan,In Vitro Transporter Inhibition
3,napdi_219,http://napdi.org/napdi_srs_imports:hydrastis_c...,goldenseal root extract,http://purl.obolibrary.org/obo/CHEBI_791,17-beta-estradiol glucuronide,positive,NaPDI,OATP1B1 (SLCO1B1),NaN,NaN,...,Hydrastis canadensis,http://napdi.org/napdi_srs_imports:hydrastis_c...,219.0,NaN,NaN,NaN,NaN,In Vitro Transporter Inhibition,PMID:nan,In Vitro Transporter Inhibition
4,napdi_41,http://purl.obolibrary.org/obo/CHEBI_31018,(-)-gallocatechin,http://purl.obolibrary.org/obo/CHEBI_17224,4-methylumbelliferone,negative,NaPDI,UGT,NaN,NaN,...,Camellia sinensis,http://napdi.org/napdi_srs_imports:camellia_si...,41.0,NaN,NaN,NaN,NaN,In Vitro Enzyme Negligible Inhibition,PMID:29467215,In Vitro Enzyme Inhibition


In [22]:
uri_to_curie_map_file = '/home/sanya/np-kg/resources/knowledge_graphs/uri_to_curie_map.tsv'
##load as dictionary
uri_to_curie_map = {}
with open(uri_to_curie_map_file, 'r') as f:
    for line in f:
        uri, curie = line.strip().split('\t')
        uri_to_curie_map[uri] = curie
len(uri_to_curie_map)

1089472

In [23]:
for key in uri_to_curie_map:
    print(key, uri_to_curie_map[key])
    break

http://purl.obolibrary.org/obo/CLO_0035452 clo:0035452


In [24]:
##add missing curies
uri_to_curie_map['http://purl.obolibrary.org/obo/CHEBI:9654'] = 'chebi:9654'

In [25]:
edge_df = pd.DataFrame(columns=['source', 'target', 'interaction_type'])
for i in range(len(refdf.index)):
    np_id = refdf.at[i, 'np_id']
    drug_id = refdf.at[i, 'drug_id']
    interaction_type = refdf.at[i, 'interaction_type']
    if np_id in uri_to_curie_map and drug_id in uri_to_curie_map:
        source = uri_to_curie_map[np_id]
        target = uri_to_curie_map[drug_id]
        tempdf = pd.DataFrame({'source': [source], 'target': [target], 'interaction_type': [interaction_type]})
        edge_df = pd.concat([edge_df, tempdf], ignore_index=True)
    else:
        print('URI not found in map: {} {}'.format(np_id, drug_id))
edge_df.info()

URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_229213
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_195558
URI not found in map: http://purl.obolibrary.org/obo/CHEBI_207762 http://purl.obolibrary.org/obo/CHEBI_8772
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_230905
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_229526
URI not found in map: http://napdi.org/napdi_srs_imports:citrus_paradisi http://purl.obolibrary.org/obo/CHEBI_231601
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1341 entries, 0 to 1340
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   source            1341 non-null   object
 1   target            1341 non-null   object
 2   interaction_

In [26]:
edge_df.head()

,source,target,interaction_type
0,chebi:16118,chebi:16469,positive
1,chebi:4806,chebi:16469,positive
2,chebi:69919,chebi:791,positive
3,napdi_srs_imports:hydrastis_canadensis,chebi:791,positive
4,chebi:31018,chebi:17224,negative


In [27]:
edge_df.to_csv(DIR_OUT+'reference_data_curie_edges_20240905.tsv', sep='\t', index=False)